In [179]:
import pandas as pd
import pymysql
import numpy as np


In [180]:
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="banksight",
    port=3306,
    autocommit=True
) 

cursor = conn.cursor()
print("Connected to:", conn.db.decode())

Connected to: banksight


In [181]:
base_path = "Banksight/Queries/dataset/"

customers = pd.read_csv(base_path + "customers.csv")
accounts = pd.read_csv(base_path + "accounts.csv")
transactions = pd.read_csv(base_path + "transactions.csv")
branches = pd.read_csv(base_path + "branches.csv")
loans = pd.read_csv(base_path + "loans.csv")
credit_cards = pd.read_csv(base_path + "credit_cards.csv")
support_tickets = pd.read_csv(base_path + "support_tickets.csv")


Check Null 

In [182]:
customers.isnull().sum()
accounts.isnull().sum()
transactions.isnull().sum()
branches.isnull().sum()
loans.isnull().sum()
credit_cards.isnull().sum()
support_tickets.isnull().sum()

Ticket_ID               0
Customer_ID             0
Account_ID              0
Loan_ID               371
Branch_Name             0
Issue_Category          0
Description             0
Date_Opened             0
Date_Closed           104
Priority                0
Status                  0
Resolution_Remarks      0
Support_Agent           0
Channel                 0
Customer_Rating         0
dtype: int64

In [183]:
#Remove Duplicates
for df in [customers, accounts, transactions, branches, loans, credit_cards, support_tickets]:
    df.drop_duplicates(inplace=True)

In [184]:
#Handle Nulls

datasets = [customers, accounts, transactions, branches, loans, credit_cards, support_tickets]

for df in datasets:
    df.replace(['', ' ', 'NA', 'null'], pd.NA, inplace=True)

In [185]:
customers['join_date'] = pd.to_datetime(
    customers['join_date'], 
    errors='coerce'
)

# Convert to MySQL format
customers['join_date'] = customers['join_date'].dt.strftime('%Y-%m-%d')

date_columns = [
    (customers, ['join_date']),
    (accounts, ['last_updated']),
    (transactions, ['txn_time']),
    (loans, ['Start_Date', 'End_Date']),
    (credit_cards, ['Issued_Date', 'Expiry_Date']),
    (support_tickets, ['Date_Opened', 'Date_Closed'])
]

for df, cols in date_columns:
    for col in cols:
        df[col] = df[col].where(df[col].notna(), None)

In [186]:
#Fix Customer_ID Format

# Ensure all IDs are strings
customers['customer_id'] = customers['customer_id'].astype(str)
accounts['customer_id'] = accounts['customer_id'].astype(str)
transactions['customer_id'] = transactions['customer_id'].astype(str)
loans['Customer_ID'] = 'C' + loans['Customer_ID'].astype(str).str.zfill(4)
credit_cards['Customer_ID'] = 'C' + credit_cards['Customer_ID'].astype(str).str.zfill(4)
support_tickets['Customer_ID'] = support_tickets['Customer_ID'].astype(str)

In [187]:
#Remove Invalid Foreign Keys

accounts = accounts[accounts['customer_id'].isin(customers['customer_id'])]
support_tickets = support_tickets[support_tickets['Customer_ID'].isin(customers['customer_id'])]
transactions = transactions[transactions['customer_id'].isin(customers['customer_id'])]


In [188]:
#Fix Outliers

# Age
customers = customers[(customers['age'] >= 18) & (customers['age'] <= 100)]

#transaction Amount
removed_outliers = transactions[
    transactions['amount'] >= 1000000
]

print("Outliers removed:", len(removed_outliers))

Outliers removed: 0


In [189]:
#Create Tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(10) PRIMARY KEY,
    name VARCHAR(50),
    gender VARCHAR(10),
    age INT,
    city VARCHAR(50),
    account_type VARCHAR(20),
    join_date DATE
)
          """)
conn.commit()

cursor.execute("""
CREATE TABLE IF NOT EXISTS accounts (
    customer_id VARCHAR(10) PRIMARY KEY,
    account_balance DECIMAL(15,2),
    last_updated DATETIME,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    txn_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(10),
    txn_type VARCHAR(50),
    amount DECIMAL(15,2),
    txn_time DATETIME,
    status VARCHAR(20),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS branches (
    Branch_ID VARCHAR(10) PRIMARY KEY,
    Branch_Name VARCHAR(50) NOT NULL,
    City VARCHAR(50),
    Manager_Name VARCHAR(50),
    Total_Employees INT,
    Branch_Revenue DECIMAL(15,2),
    Opening_Date DATE,
    Performance_Rating INT
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS loans (
    Loan_ID INT PRIMARY KEY,
    Customer_ID VARCHAR(10),
    Account_ID VARCHAR(50),
    Branch VARCHAR(100),
    Loan_Type VARCHAR(50),
    Loan_Amount DECIMAL(15,2),
    Interest_Rate DECIMAL(5,2),
    Loan_Term_Months INT,
    Start_Date DATE,
    End_Date DATE,
    Loan_Status VARCHAR(30),
    FOREIGN KEY (Customer_ID) REFERENCES customers(customer_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS credit_cards (
    Card_ID INT PRIMARY KEY,
    Customer_ID VARCHAR(10),
    Account_ID VARCHAR(50),
    Branch VARCHAR(100),
    Card_Number VARCHAR(30),
    Card_Type VARCHAR(50),
    Card_Network VARCHAR(50),
    Credit_Limit DECIMAL(15,2),
    Current_Balance DECIMAL(15,2),
    Issued_Date DATE,
    Expiry_Date DATE,
    Status VARCHAR(50),
    FOREIGN KEY (Customer_ID) REFERENCES customers(customer_id)
)

""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS support_tickets (
    Ticket_ID VARCHAR(50) PRIMARY KEY,
    Customer_ID VARCHAR(10),
    Account_ID VARCHAR(50),
    Loan_ID VARCHAR(50),
    Branch_Name VARCHAR(100),
    Issue_Category VARCHAR(100),
    Description TEXT,
    Date_Opened DATETIME,
    Date_Closed DATETIME,
    Priority VARCHAR(20),
    Status VARCHAR(30),
    Resolution_Remarks TEXT,
    Support_Agent VARCHAR(100),
    Channel VARCHAR(50),
    Customer_Rating INT,
    FOREIGN KEY (Customer_ID) REFERENCES customers(customer_id)
)
""")

0

In [190]:
#Reset Tables

cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

tables = ["transactions","accounts","loans","credit_cards","support_tickets","customers","branches"]

for table in tables:
    cursor.execute(f"TRUNCATE TABLE {table}")

cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
conn.commit()

In [191]:
#Insert Data

cursor.executemany("""
INSERT INTO customers 
(customer_id, name, gender, age, city, account_type, join_date)
VALUES (%s,%s,%s,%s,%s,%s,%s)
""", customers.values.tolist())

cursor.executemany("""
INSERT INTO accounts 
(customer_id, account_balance, last_updated)
VALUES (%s,%s,%s)
""", accounts.values.tolist())

cursor.executemany("""
INSERT INTO transactions 
(txn_id, customer_id, txn_type, amount, txn_time, status)
VALUES (%s,%s,%s,%s,%s,%s)
""", transactions.values.tolist())

cursor.executemany("""
INSERT INTO branches 
(Branch_ID, Branch_Name, City, Manager_Name,
 Total_Employees, Branch_Revenue, Opening_Date, Performance_Rating)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
""", branches.values.tolist())

#  FIXED loans insert
cursor.executemany("""
INSERT INTO loans 
(Loan_ID, Customer_ID, Account_ID, Branch, Loan_Type,
 Loan_Amount, Interest_Rate, Loan_Term_Months,
 Start_Date, End_Date, Loan_Status)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", loans.values.tolist())

#  credit cards
cursor.executemany("""
INSERT INTO credit_cards 
(Card_ID, Customer_ID, Account_ID, Branch,
 Card_Number, Card_Type, Card_Network,
 Credit_Limit, Current_Balance,
 Issued_Date, Expiry_Date, Status)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", credit_cards.values.tolist())

#  support tickets
data = [
    tuple(None if pd.isna(x) else x for x in row)
    for row in support_tickets.to_numpy()
]

cursor.executemany("""
INSERT INTO support_tickets 
(Ticket_ID, Customer_ID, Account_ID, Loan_ID,
 Branch_Name, Issue_Category, Description,
 Date_Opened, Date_Closed, Priority,
 Status, Resolution_Remarks, Support_Agent,
 Channel, Customer_Rating)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", data)

conn.commit()
print("✅ All data inserted successfully!")

✅ All data inserted successfully!


Question 1: How many customers exist per city, and what is their average account balance?

In [192]:
query_1= """
SELECT 
    c.city,
    COUNT(DISTINCT c.customer_id) AS total_customers,  
    AVG(a.account_balance) AS average_balance
FROM customers c
LEFT JOIN accounts a
    ON c.customer_id = a.customer_id
GROUP BY c.city;
"""

In [193]:
df = pd.read_sql(query_1, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\1149281135.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_1, conn)


,city,total_customers,average_balance
0,Adamtown,1,45424.11
1,Alanton,1,392508.77
2,Albertchester,1,387844.67
3,Alexanderstad,1,383016.76
4,Alexfurt,1,403292.20
...,...,...,...
481,Wintershaven,1,455365.96
482,Woodfort,1,434936.56
483,Yanghaven,1,163152.58
484,Zacharychester,1,311439.87


Question 2 : Which account type (Savings, Current, Loan, etc.) holds the highest total balance?

In [194]:
query_2 = """
SELECT 
        c.account_type,
        SUM(a.account_balance) AS total_balance
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    GROUP BY c.account_type
    ORDER BY total_balance DESC
    LIMIT 5;
  """

In [195]:
df = pd.read_sql(query_2, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\646018716.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_2, conn)


,account_type,total_balance
0,Current,61617575.41
1,Savings,61586753.31


Question 3: Who are the top 10 customers by total account balance across all account types?

In [196]:
query_3 = """
SELECT 
    c.customer_id,
    c.name,
    SUM(a.account_balance) AS total_balance
FROM customers c
JOIN accounts a
    ON c.customer_id = a.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_balance DESC
LIMIT 10;
"""


In [197]:
df = pd.read_sql(query_3, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\3857336370.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_3, conn)


,customer_id,name,total_balance
0,C0314,Amy Nguyen,498739.49
1,C0011,Carolyn Higgins,498317.41
2,C0111,Lisa Russell,497040.60
3,C0392,Amber Williams,494779.97
4,C0077,Audrey Peck,494699.91
5,C0499,Alvin Compton,484758.19
6,C0338,Sharon Bennett DVM,482746.37
7,C0317,Denise Campbell,481746.58
8,C0220,Jeffrey Smith,481568.14
9,C0146,Ann Crosby,479706.41


Question 4: Which customers opened accounts in 2023 with a balance above ₹1,00,000?

In [198]:
query_4= """
SELECT 
    c.customer_id,
    c.name,
    c.join_date,
    a.account_balance
FROM accounts a
JOIN customers c
    ON a.customer_id = c.customer_id
WHERE YEAR(c.join_date) = 2023
  AND a.account_balance > 100000;
"""


In [199]:
df = pd.read_sql(query_4, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\1502449166.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_4, conn)


,customer_id,name,join_date,account_balance
0,C0002,Holly Parker,2023-12-08,150735.15
1,C0007,Jasmine Morris,2023-10-09,291736.90
2,C0017,Jesus Davenport,2023-06-07,345239.48
3,C0021,William Jones,2023-04-01,185053.68
4,C0023,Jennifer Hodges,2023-02-11,447170.74
...,...,...,...,...
107,C0478,Ryan Jackson,2023-04-18,400751.68
108,C0484,Rodney Cruz,2023-03-09,150039.17
109,C0489,Andrea Austin,2023-01-28,407686.39
110,C0499,Alvin Compton,2023-06-16,484758.19


Question 5: What is the total transaction volume (sum of amounts) by transaction type?

In [200]:
query_5= """
SELECT 
    txn_type,
    SUM(amount) AS total_volume
FROM transactions
GROUP BY txn_type;
"""


In [201]:
df = pd.read_sql(query_5, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\2785224353.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_5, conn)


,txn_type,total_volume
0,deposit,1.969798e+08
1,withdrawal,1.509890e+08
2,transfer,5.059180e+07
3,online fraud,2.382950e+07
4,purchase,7.375031e+07


Question 6: How many failed transactions occurred for each transaction type?

In [202]:
query_6 = """
SELECT 
    txn_type,
    COUNT(*) AS failed_count
FROM transactions
WHERE status = 'failed'
GROUP BY txn_type;
"""


In [203]:
df = pd.read_sql(query_6, conn)
df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\304927803.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_6, conn)


,txn_type,failed_count
0,online fraud,489
1,deposit,430
2,withdrawal,313
3,purchase,167
4,transfer,94


Question 7: What is the total number of transactions per transaction type?

In [204]:
query_7= """
SELECT 
    txn_type,
    COUNT(*) AS total_transactions
FROM transactions
GROUP BY txn_type;
"""


In [205]:
df = pd.read_sql(query_7, conn)
df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\2447116253.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_7, conn)


,txn_type,total_transactions
0,deposit,3909
1,withdrawal,3058
2,transfer,1011
3,online fraud,489
4,purchase,1510


Question 8: Which accounts have 5 or more high-value transactions above ₹20,000?

In [206]:
query_8 = """
SELECT 
    customer_id,
    COUNT(*) AS high_value_txn
FROM transactions
WHERE amount > 20000
GROUP BY customer_id
HAVING COUNT(*) >= 5;
"""


In [207]:
df = pd.read_sql(query_8, conn)
df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\644663087.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_8, conn)


,customer_id,high_value_txn
0,C0002,9
1,C0003,15
2,C0004,11
3,C0005,11
4,C0006,17
...,...,...
494,C0496,25
495,C0497,17
496,C0498,13
497,C0499,17


Question 9: What is the average loan amount and interest rate by loan type (Personal, Auto, Home, etc.)?


In [208]:
query_9= """
SELECT 
    Loan_Type,
    AVG(Loan_Amount) AS avg_Loan_Amount,
    AVG(Interest_Rate) AS avg_Interest_Rate
FROM loans
GROUP BY Loan_Type;

"""


In [209]:
df = pd.read_sql(query_9, conn)
df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\3006328155.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_9, conn)


,Loan_Type,avg_Loan_Amount,avg_Interest_Rate
0,Personal,2.461544e+06,10.463391
1,Business,2.793969e+06,10.625437
2,Auto,2.399261e+06,10.532523
3,Home,2.344577e+06,10.872479
4,Education,2.393948e+06,10.114393


Question 10: Which customers currently hold more than one active or approved loan?


In [210]:
query_10 = """
SELECT 
    customer_id,
    COUNT(*) AS total_loans
FROM loans
WHERE loan_status IN ('Active', 'Approved')
GROUP BY customer_id
HAVING COUNT(*) > 1;
"""


In [211]:
df = pd.read_sql(query_10, conn)
df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\17886988.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_10, conn)


,customer_id,total_loans
0,C1005,2
1,C1010,4
2,C1020,2
3,C1059,2
4,C1073,2
5,C1088,3
6,C1101,2
7,C1108,2
8,C1123,2
9,C1126,2


Question 11: Who are the top 5 customers with the highest outstanding (non-closed) loan amounts?

In [212]:
query_11 = """
SELECT 
    c.customer_id,
    c.name,
    SUM(l.loan_amount) AS total_outstanding_loan
FROM loans l
JOIN customers c
    ON CAST(SUBSTR(c.customer_id, 2) AS UNSIGNED) = l.customer_id
WHERE l.loan_status <> 'Closed'
GROUP BY c.customer_id, c.name
ORDER BY total_outstanding_loan DESC
LIMIT 5;
"""


In [213]:
df_top_loans = pd.read_sql(query_11, conn)
df_top_loans

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\623369087.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_loans = pd.read_sql(query_11, conn)


,customer_id,name,total_outstanding_loan


Question 12: What is the average loan amount per branch?


In [214]:
query_12 = """
SELECT
    branch,
    AVG(loan_amount) AS avg_loan_amount
FROM loans
GROUP BY branch
ORDER BY avg_loan_amount DESC;
"""




In [215]:
df = pd.read_sql(query_12, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\3258492921.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_12, conn)


,branch,avg_loan_amount
0,"Johnson, Arias and Kramer Branch",4995813.0
1,"Lopez, Santana and Johnson Branch",4987450.0
2,"Williams, Mcfarland and Alvarez Branch",4966720.0
3,Rogers Group Branch,4890404.0
4,Burke-Moreno Branch,4864325.0
...,...,...
339,"Lee, Escobar and Torres Branch",206282.0
340,Black Inc Branch,157126.0
341,Russell-Patel Branch,131137.0
342,Johnson-Hunter Branch,93465.0


Question 13: How many customers exist in each age group (e.g., 18–25, 26–35, etc.)?

In [216]:
query_13 = """
SELECT
    CASE
        WHEN age BETWEEN 18 AND 25 THEN '18–25'
        WHEN age BETWEEN 26 AND 35 THEN '26–35'
        WHEN age BETWEEN 36 AND 45 THEN '36–45'
        WHEN age BETWEEN 46 AND 55 THEN '46–55'
        WHEN age >= 56 THEN '56+'
        ELSE 'Unknown'
    END AS age_group,
    COUNT(*) AS total_customers
FROM customers
GROUP BY age_group
ORDER BY age_group;
"""


In [217]:
df = pd.read_sql(query_13, conn)
df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\1352469112.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_13, conn)


,age_group,total_customers
0,18–25,79
1,26–35,76
2,36–45,101
3,46–55,100
4,56+,144


Question 14: Which issue categories have the longest average resolution time?


In [218]:
query_14 = """
SELECT
    issue_category,
    AVG(DATEDIFF(date_closed, date_opened)) AS avg_resolution_days
FROM support_tickets
WHERE date_closed IS NOT NULL
  AND date_opened IS NOT NULL
GROUP BY issue_category
ORDER BY avg_resolution_days DESC;
"""


In [219]:
df = pd.read_sql(query_14, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\2002975064.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_14, conn)


,issue_category,avg_resolution_days
0,Card Not Working,18.5000
1,Mismatch in Account Statement,17.9032
2,Debit Card Delivery Delay,17.8148
3,UPI Payment Failed,17.4583
4,Account Frozen,17.1154
5,Account Closure Request,16.9333
6,Loan Foreclosure,16.5833
7,Cheque Bounce,16.0000
8,EMI Auto-debit Failed,15.9762
9,Online Transfer Failed,15.4762


Question 15: Which support agents have resolved the most critical tickets with high customer ratings (≥4)?


In [220]:
query_15 = """
SELECT
    support_agent,
    COUNT(*) AS resolved_critical_tickets
FROM support_tickets
WHERE priority = 'Critical'
  AND customer_rating >= 4
  AND status IN ('Resolved', 'Closed')
GROUP BY support_agent
ORDER BY resolved_critical_tickets DESC;
"""


In [221]:
df = pd.read_sql(query_15, conn)
df


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_528\1530480359.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_15, conn)


,support_agent,resolved_critical_tickets
0,Gabriel Mutti,3
1,Owen Mahajan,3
2,Chanakya Natt,3
3,Wishi Chhabra,2
4,Veda Samra,2
5,Wriddhish Sarraf,2
6,Vansha Sood,2
7,Udyati Mody,1
8,Ekanta Devan,1
9,Rehaan Balakrishnan,1
